# CalStateTesting_Example

Notebook companion to `generate_calstate_testing.py` -- a Python port of Jeff's original PHP/MySQL generator for the California STAR standardized-test-score visualization (San Diego Unified School District, 2003-2012).

**Glyph design:** a central wireframe sphere represents one K-12 school (~50 in the district for a given year); a thin torus anchored to it carries 6 colored demographic-subgroup branches (Males / Females / Black / Asian / Hispanic / Caucasian), each sized by how many students were tested; each subgroup fans out into per-test branches sized by mean scale score and colored by percent-above-proficient banding.

**4-level hierarchy** (branch_level 0-3):
- **BL0** one node per (school, year) -- Sphere-Wire, positioned by real lat/lon
- **BL1** torus anchor, child of BL0
- **BL2** 6 demographic subgroups, child of BL1, sized by `2*log(1 + total_subgroup_tested/25)`
- **BL3** up to 25 real tests per subgroup, sized by `mean_scale_score/600`, colored by proficiency band. Octahedron geometry (changed from the PHP's literal Torus default -- Torus tanked framerate at full multi-year scale).

This notebook is for interactively tweaking parameters and previewing output; for batch/CI use, run `generate_calstate_testing.py` directly from the command line, or edit the two constants in `run_full_dataset.py`.

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from generate_calstate_testing import build_arg_parser, generate

## Parameters

Edit this list and re-run -- it's the exact same flags `generate_calstate_testing.py` takes on the command line (run `python generate_calstate_testing.py --help` for the full list: `--disadvantaged`, `--max-branch-level`, `--bl2-topo`, `--exclude-groups`, `--prism`, `--geom`, `--rotation`, `--dbquery-flag`, ...).

`--source mysql` requires a reachable MySQL server with the `calstatetesting_all` database (`--db-host`/`--db-user`/`--db-password`/`--db-name`, default `localhost`/`root`/*(empty)*/`calstatetesting_all`, matching the original PHP's connection). `--source csv` instead reads `sdusd<year>.csv` / `schoollatlng.csv` / `schoolnames.csv` from `--data-dir`.

In [ ]:
cli_args = [
    "--source", "mysql",
    "--grade", "8",
    "--start-year", "2003",
    "--end-year", "2012",
    "--prefix", "CalStateTesting_Example_notebook",
]

args = build_arg_parser().parse_args(cli_args)
args

## Generate the node/tag CSVs

In [ ]:
node_path, tag_path = generate(args)
node_path, tag_path

## Peek at the output

In [ ]:
import pandas as pd

nodes_df = pd.read_csv(node_path)
tags_df = pd.read_csv(tag_path)

print("Nodes per branch level:")
print(nodes_df["branch_level"].value_counts().sort_index())
tags_df.head(10)

## Optional: render a quick preview

Uses the same headless-GL pattern as `tools/export_frame.py` (loads the CSV through the real `Viewport`, grabs a framebuffer -- no app window needed). Requires PySide6 + the `glyphviz_gl`/`glyphviz_core` packages to be importable, i.e. run this notebook's kernel from the repo root (or add the repo root to `sys.path` below).

In [ ]:
REPO_ROOT = HERE.parent.parent  # examples/CalStateTesting_Example -> repo root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from PySide6.QtWidgets import QApplication

app = QApplication.instance() or QApplication(sys.argv[:1])

from glyphviz_core.csv_reader import load_node_csv
from glyphviz_gl.viewport import Viewport

nodes = load_node_csv(str(node_path))
vp = Viewport()
vp.set_nodes(nodes)

preview_path = HERE / "notebook_preview.png"
vp.export_png(str(preview_path), width=960, height=540)

from IPython.display import Image
Image(filename=str(preview_path))

## Load in GlyphViz

File > Open Node CSV -> the `node_path` printed above (e.g. `CalStateTesting_Example_notebook_gv_node.csv`).